# Private first-fold scratch baseline

Output-free, GPU-only Kaggle batch entry point. The fixed run is disabled by default; the authorized operator must explicitly enable it before Save Version → Save & Run All. The cell audits the mounted package and validates a plan before any fit. It performs no installation or download.

In [ ]:
"""Private, output-free first-fold Kaggle batch entry point."""
import base64
import hashlib
import json
import math
import re
import sys
from pathlib import Path, PurePosixPath

RUN_TRAINING = False
RUN_CONFIG = {
    "mode": "fold",
    "manifest_path": "data/private/500-study-training-cohort.json",
    "folds_path": "data/private/500-study-training-folds.json",
    "image_root": "/kaggle/input/competitions/rsna-knee-abnormality-detection/train_series",
    "output_dir": "/kaggle/working/kaggle-first-fold-baseline/run01",
    "fold": 0,
    "seed": 42,
    "epochs": 5,
    "max_steps": None,
    "overfit_limit": None,
    "output_size": 256,
    "max_slices": 8,
    "device": "cuda",
    "learning_rate": 0.001,
    "feature_width": 64,
    "expected_member_count": 24,
    "expected_manifest_sha256": "e88c61c22a463737f16951b7c76db03834034aecd1b31dc61b4ce495080b2be2",
    "expected_folds_sha256": "b82b376b0086201e3364b11a0203b62890e628442075a52ea4a9e09ee0e778ac",
    "expected_inventory_sha256": "5ed26739dbec2d25a398c08438fbdcab7db4ba2d3dc60d9844b7dd54058f6807",
    "expected_archive_provenance": "fb803d396d6afaf3ad31031b41abd4e8805d961408b1f4377749dfc777ce36af",
    "expected_training_count": 333,
    "expected_validation_count": 167,
    "expected_steps": 1665,
    "private_input_root": "/kaggle/input/datasets/davidsulleiro/rsna-knee-five-study-reader-private",
}
EXPECTED_INVENTORY_B64 = "ew0KICAiZXhjbHVzaW9ucyI6IFsNCiAgICAiYnl0ZWNvZGUiLA0KICAgICJyZXBvcnRzIiwNCiAgICAiZXZpZGVuY2UiLA0KICAgICJpbWFnZXMiLA0KICAgICJjaGVja3BvaW50cyIsDQogICAgIm91dHB1dHMiLA0KICAgICJjcmVkZW50aWFscyINCiAgXSwNCiAgImZvbGRzX3NoYTI1NiI6ICJiODJiMzc2YjAwODYyMDFlMzM2NGIxMWEwMjAzYjYyODkwZTYyODQ0MjA3NWE1MmVhNGE5ZTA5ZWUwZTc3OGFjIiwNCiAgIm1hbmlmZXN0X3NoYTI1NiI6ICJlODhjNjFjMjJhNDYzNzM3ZjE2OTUxYjdjNzZkYjAzODM0MDM0YWVjZDFiMzFkYzYxYjRjZTQ5NTA4MGIyYmUyIiwNCiAgIm1lbWJlcnMiOiB7DQogICAgImNvbmZpZ3Mva2FnZ2xlX2VpZ2h0X3N0dWR5X292ZXJmaXQuanNvbiI6ICJjYzM3ZmE4ZDdhZjY1YTNlN2Q1ZjliZGE1ZDQyYTcwODA2ZjA4ZTFlMTA2ZThiMWVmNmZkNGE5MmJkZWEyM2E1IiwNCiAgICAiZGF0YS9wcml2YXRlLzUwMC1zdHVkeS10cmFpbmluZy1jb2hvcnQuanNvbiI6ICJlODhjNjFjMjJhNDYzNzM3ZjE2OTUxYjdjNzZkYjAzODM0MDM0YWVjZDFiMzFkYzYxYjRjZTQ5NTA4MGIyYmUyIiwNCiAgICAiZGF0YS9wcml2YXRlLzUwMC1zdHVkeS10cmFpbmluZy1mb2xkcy5qc29uIjogImI4MmIzNzZiMDA4NjIwMWUzMzY0YjExYTAyMDNiNjI4OTBlNjI4NDQyMDc1YTUyZWE0YTllMDllZTBlNzc4YWMiLA0KICAgICJub3RlYm9va3Mva2FnZ2xlX2VpZ2h0X3N0dWR5X292ZXJmaXQuaXB5bmIiOiAiN2Y3ODUyNTUwYzk2NDk3MjU5M2U1MjI4YWEyZDA4YzFkMzNiMDUyNzA2M2NlYjcwZjFmNjE0YzZmMjBlM2EyOSIsDQogICAgInNyYy9yc25hX2tuZWUvX19pbml0X18ucHkiOiAiZTNiMGM0NDI5OGZjMWMxNDlhZmJmNGM4OTk2ZmI5MjQyN2FlNDFlNDY0OWI5MzRjYTQ5NTk5MWI3ODUyYjg1NSIsDQogICAgInNyYy9yc25hX2tuZWUvZGF0YS9fX2luaXRfXy5weSI6ICJlM2IwYzQ0Mjk4ZmMxYzE0OWFmYmY0Yzg5OTZmYjkyNDI3YWU0MWU0NjQ5YjkzNGNhNDk1OTkxYjc4NTJiODU1IiwNCiAgICAic3JjL3JzbmFfa25lZS9kYXRhL2NvaG9ydC5weSI6ICJmY2EyZjNiMDk2MTBjODUwZDhkNTAwYjlhOTQyZmIxZDEwOWJiNTk5NmI4OWM2ZGY5MzIyOWYzMTZhMTQyOGU2IiwNCiAgICAic3JjL3JzbmFfa25lZS9kYXRhL2RpY29tX2JlbmNobWFyay5weSI6ICI4ZDU5Yzk3YmY4NWMzYzQ2ZmFiNTRjZGJkMmJiOGJlYWE4YzM2ZmU1ZmVmMjlmN2JiNjNiNmYzMzJlN2U3ODNmIiwNCiAgICAic3JjL3JzbmFfa25lZS9kYXRhL2RpY29tX3ByZXByb2Nlc3NpbmcucHkiOiAiMTRlZDQxOTUxNTQyZTYwZTFhYjQ4MTVkN2I5ZTA3NTY5YWVlMzhlMzg4N2IyZTI3MzZkNmIxM2M3YmIzNDZmZiIsDQogICAgInNyYy9yc25hX2tuZWUvZGF0YS9wcmVwcm9jZXNzaW5nLnB5IjogImRjNjU4OWFjMjRiNGQ2YzlmMDZlZjM0NDRiNDk2NmIxNzg4ZjAwMzJhNjdiNjJlYTRkYTkwN2UyY2Y0NDU0YzYiLA0KICAgICJzcmMvcnNuYV9rbmVlL2RhdGEvdHJhaW5pbmdfZGF0YXNldC5weSI6ICJlNmU2N2M5M2MwMWY4Yzg3N2FiNDE2NTY4MTliNzlkMGE0ZWY5MTA5ZDY2Zjc4MDRkMTIwMTg3YzRjOWE2ZGIyIiwNCiAgICAic3JjL3JzbmFfa25lZS9sYWJlbHMvX19pbml0X18ucHkiOiAiZTNiMGM0NDI5OGZjMWMxNDlhZmJmNGM4OTk2ZmI5MjQyN2FlNDFlNDY0OWI5MzRjYTQ5NTk5MWI3ODUyYjg1NSIsDQogICAgInNyYy9yc25hX2tuZWUvbGFiZWxzL3Byb21wdF9jb21wYXJpc29uLnB5IjogImFlN2RiOWY3MDA5OWRjMzkwYjQxNzE5Y2M4M2UyMmY3MWE2YmIyZGI1MDA2ZWEwZjk1OWE2NzQyY2Y3MmIxNjgiLA0KICAgICJzcmMvcnNuYV9rbmVlL2xhYmVscy9zaXdjX2F1dGgucHkiOiAiZWU3YTNkNWNiZjQ4MjcwMDQwMmFhYmZiMjA5ZDVjZGRlOTlmODc1NDIzNjVjOWVkZmNjOTg2ODliZGVlMGVmMSIsDQogICAgInNyYy9yc25hX2tuZWUvbGFiZWxzL3Npd2NfcmVzcG9uc2VzLnB5IjogIjZhMWFlOGZhYzg4NDYyZTllNzNkNTNhMjFiOTJiM2QxZDBiMzYwYzAzOTU1MjdhZTIxZjQyNDIxZmRmMWRiZTMiLA0KICAgICJzcmMvcnNuYV9rbmVlL2xhYmVscy9zdWJzY3JpcHRpb24ucHkiOiAiNDYyMDliOTg1ZTRlOGMxZWNhYzBhYWQyYTRlNzIwN2UyMjViYjVkZDE2ZWNjMDAzMWVkY2Y0NmExMzBiZmY4OCIsDQogICAgInNyYy9yc25hX2tuZWUvbW9kZWxzL19faW5pdF9fLnB5IjogImJjMDA4MTU5ODA3MzQ5NjJlNzg5YWRlZDc1N2FlNmY4MDdkOTFjMTVlYmRmYzg2OGI1MWY0NTZlZjEyZTk2MGQiLA0KICAgICJzcmMvcnNuYV9rbmVlL21vZGVscy9iYXNlbGluZS5weSI6ICI2MTRjZWM1ZmM1NDg4NzQwNmFjN2Y0NzBjYzk4NzlhMzQ0N2M3N2JmZWJiMTM5MzBmZmM0ZWI1NGE4M2JkZmEyIiwNCiAgICAic3JjL3JzbmFfa25lZS90cmFpbmluZy9fX2luaXRfXy5weSI6ICJlMjMzMTY3NzNjZGE0Yzk3ZWY5Y2ZiZjg2NDA2OTFhYWM0NTZiMDNhMmRkMDJiMGZmMGZmNDQ3YTY2MjFiYWI4IiwNCiAgICAic3JjL3JzbmFfa25lZS90cmFpbmluZy9iYXNlbGluZS5weSI6ICI1ZDkzMmY3ZGZhNGQ2NDhmNTk1MjJiYTVhOTI2MDkxNTVlMjViMTA2YzZlM2QzMjg0N2E0ZmRmMzk1ZGMxM2I3IiwNCiAgICAic3JjL3JzbmFfa25lZS92YWxpZGF0aW9uL19faW5pdF9fLnB5IjogImUzYjBjNDQyOThmYzFjMTQ5YWZiZjRjODk5NmZiOTI0MjdhZTQxZTQ2NDliOTM0Y2E0OTU5OTFiNzg1MmI4NTUiLA0KICAgICJzcmMvcnNuYV9rbmVlL3ZhbGlkYXRpb24vY29ob3J0LnB5IjogIjM0Y2M3MGM1MzI2OTBjNTQ2OWY4NDE5ZjdmZGE5ZThjMTIzNmZlY2IxMTVjNTgzOGQyOTAzNGNiMmU2YjNjYjIiLA0KICAgICJzcmMvcnNuYV9rbmVlL3ZhbGlkYXRpb24vbWV0cmljcy5weSI6ICI1MGUxNGViNGViZjJkMDUxZGNhYTU5YTRmYjdhMzlkYzlkZTAyODgwODE5MDhkZDhlZmZiMTg1Yzk4OGEyMmJlIiwNCiAgICAic3JjL3JzbmFfa25lZS92YWxpZGF0aW9uL211bHRpbGFiZWwucHkiOiAiY2Q2MzlmZmZhODA5Y2I3NDc2Y2RhODRmOTBlZmJmZGJlZGJmMWUwMmY0MWQ3ZTUzNzA2Mzk0MTM3OGNjODViMCINCiAgfSwNCiAgInBhY2thZ2Vfc2hhMjU2IjogImZiODAzZDM5NmQ2YWZhZjNhZDMxMDMxYjQxYWJkNGU4ODA1ZDk2MTQwOGIxZjQzNzc3NDlkZmM3NzdjZTM2YWYiLA0KICAic2NoZW1hX3ZlcnNpb24iOiAia2FnZ2xlLXRyYWluaW5nLXBhY2thZ2UtaW52ZW50b3J5LXYxIg0KfQ0K"
SAFE_REASON_CODES = frozenset({
    "inventory_invalid", "inventory_hash_mismatch", "inventory_contract_mismatch",
    "inventory_member_invalid", "package_member_unavailable", "package_member_hash_mismatch",
    "package_inventory_unavailable", "package_membership_mismatch", "private_data_hash_mismatch",
    "mounted_input_unavailable", "plan_contract_mismatch", "plan_count_mismatch",
    "plan_hash_mismatch", "plan_step_count_mismatch", "training_flag_invalid",
    "cuda_unavailable", "input_unavailable", "output_must_be_new", "output_path_conflict",
    "output_path_invalid", "config_invalid", "cuda_determinism_unavailable",
    "determinism_unavailable", "model_output_invalid", "loss_invalid", "training_subset_empty",
    "output_write_failed", "execute_flag_invalid", "training_dependencies_unavailable",
    "training_partition_empty", "validation_partition_empty", "dataset_validation_failed",
    "no_training_steps", "prediction_invalid", "checkpoint_write_failed", "training_failed",
    "runtime_failed", "training_report_incomplete",
})

def safe_failure_diagnostics(error):
    candidate = getattr(error, "diagnostics", None)
    try:
        from rsna_knee.training.baseline import _validate_failure_diagnostics
    except ImportError:
        return None
    return _validate_failure_diagnostics(candidate)


def _sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def audit_package(input_root, inventory_bytes=None, *, expected_inventory_sha256=None,
                  expected_manifest_sha256=None, expected_folds_sha256=None,
                  expected_archive_provenance=None):
    """Verify every unpacked package byte before adding its source to sys.path."""
    config = RUN_CONFIG
    expected_inventory_sha256 = expected_inventory_sha256 or config["expected_inventory_sha256"]
    expected_manifest_sha256 = expected_manifest_sha256 or config["expected_manifest_sha256"]
    expected_folds_sha256 = expected_folds_sha256 or config["expected_folds_sha256"]
    expected_archive_provenance = expected_archive_provenance or config["expected_archive_provenance"]
    root = Path(input_root).resolve(strict=True)
    if inventory_bytes is None:
        try:
            inventory_bytes = base64.b64decode(EXPECTED_INVENTORY_B64, validate=True)
        except (ValueError, TypeError):
            raise RuntimeError("inventory_invalid") from None
    if hashlib.sha256(inventory_bytes).hexdigest() != expected_inventory_sha256:
        raise RuntimeError("inventory_hash_mismatch")
    try:
        inventory = json.loads(inventory_bytes.decode("utf-8"))
    except (UnicodeError, json.JSONDecodeError):
        raise RuntimeError("inventory_invalid") from None
    members = inventory.get("members") if isinstance(inventory, dict) else None
    if (inventory.get("schema_version") != "kaggle-training-package-inventory-v1"
            or inventory.get("package_sha256") != expected_archive_provenance
            or inventory.get("manifest_sha256") != expected_manifest_sha256
            or inventory.get("folds_sha256") != expected_folds_sha256
            or not isinstance(members, dict)
            or len(members) != config["expected_member_count"]):
        raise RuntimeError("inventory_contract_mismatch")
    expected_paths = set()
    for relative, expected_hash in members.items():
        rel = PurePosixPath(relative) if isinstance(relative, str) else None
        if (rel is None or rel.is_absolute() or ".." in rel.parts or chr(92) in relative
                or not isinstance(expected_hash, str)
                or not re.fullmatch(r"[0-9a-f]{64}", expected_hash)):
            raise RuntimeError("inventory_member_invalid")
        path = root.joinpath(*rel.parts)
        try:
            resolved = path.resolve(strict=True)
            resolved.relative_to(root)
        except (OSError, ValueError):
            raise RuntimeError("package_member_unavailable") from None
        if not resolved.is_file() or _sha256(resolved) != expected_hash:
            raise RuntimeError("package_member_hash_mismatch")
        expected_paths.add(rel.as_posix())
    actual_paths = set()
    try:
        for path in root.rglob("*"):
            if path.is_symlink():
                raise RuntimeError("package_member_invalid")
            if path.is_file():
                actual_paths.add(path.relative_to(root).as_posix())
    except OSError:
        raise RuntimeError("package_inventory_unavailable") from None
    if actual_paths != expected_paths:
        raise RuntimeError("package_membership_mismatch")
    manifest_rel = "data/private/500-study-training-cohort.json"
    folds_rel = "data/private/500-study-training-folds.json"
    if (members.get(manifest_rel) != expected_manifest_sha256
            or members.get(folds_rel) != expected_folds_sha256):
        raise RuntimeError("private_data_hash_mismatch")
    return {"validated_member_count": len(members), "manifest_matches": True,
            "folds_match": True, "remote_archive_bytes_verified": False}


def validate_plan(plan):
    expected = RUN_CONFIG
    if (not isinstance(plan, dict) or plan.get("status") != "plan_only"
            or plan.get("mode") != "fold" or plan.get("fold") != expected["fold"]):
        raise RuntimeError("plan_contract_mismatch")
    if (plan.get("training_study_count") != expected["expected_training_count"]
            or plan.get("validation_study_count") != expected["expected_validation_count"]):
        raise RuntimeError("plan_count_mismatch")
    hashes = plan.get("input_sha256")
    if (not isinstance(hashes, dict)
            or hashes.get("manifest") != expected["expected_manifest_sha256"]
            or hashes.get("folds") != expected["expected_folds_sha256"]):
        raise RuntimeError("plan_hash_mismatch")
    if expected["epochs"] * plan["training_study_count"] != expected["expected_steps"]:
        raise RuntimeError("plan_step_count_mismatch")
    return plan


def validate_completed_report(report, plan):
    """Only accept a complete, fixed-step fold result as a successful run."""
    expected_steps = RUN_CONFIG["expected_steps"]
    per_target = report.get("per_target_auc") if isinstance(report, dict) else None
    runtime = report.get("runtime") if isinstance(report, dict) else None
    metrics_ok = (
        isinstance(per_target, dict) and len(per_target) == 12
        and all(isinstance(value, (int, float)) and not isinstance(value, bool)
                and math.isfinite(value) and 0.0 <= value <= 1.0
                for value in per_target.values())
        and isinstance(report.get("macro_auc"), (int, float))
        and not isinstance(report.get("macro_auc"), bool)
        and math.isfinite(report["macro_auc"])
        and 0.0 <= report["macro_auc"] <= 1.0
    )
    if (not isinstance(report, dict) or report.get("status") != "completed"
            or report.get("mode") != plan["mode"] or report.get("fold") != plan["fold"]
            or report.get("training_study_count") != RUN_CONFIG["expected_training_count"]
            or report.get("validation_study_count") != RUN_CONFIG["expected_validation_count"]
            or report.get("training_steps_planned") != expected_steps
            or report.get("steps_completed") != expected_steps
            or report.get("input_sha256") != plan.get("input_sha256")
            or report.get("model_kind") != "scratch_study_cnn"
            or not metrics_ok
            or not isinstance(report.get("missing_class_targets"), list)
            or not isinstance(runtime, dict)
            or not isinstance(report.get("interpretation_limit"), str)):
        raise RuntimeError("training_report_incomplete")
    return report


def run_with_plan_gate(run_training_fn, config, *, run_enabled):
    if type(run_enabled) is not bool:
        raise RuntimeError("training_flag_invalid")
    plan = validate_plan(run_training_fn(config, execute=False))
    if not run_enabled:
        return plan, None
    report = run_training_fn(config, execute=True)
    return plan, validate_completed_report(report, plan)


def main():
    try:
        input_root = Path(RUN_CONFIG["private_input_root"])
        image_root = Path(RUN_CONFIG["image_root"])
        if not input_root.is_dir() or not image_root.is_dir():
            raise RuntimeError("mounted_input_unavailable")
        audit = audit_package(input_root)
        # Imports from the private package are intentionally after the byte audit.
        sys.path.insert(0, str(input_root / "src"))
        import torch
        if not torch.cuda.is_available():
            raise RuntimeError("cuda_unavailable")
        from rsna_knee.training.baseline import TrainingConfig, run_training
        values = dict(RUN_CONFIG)
        values["manifest_path"] = str(input_root / values["manifest_path"])
        values["folds_path"] = str(input_root / values["folds_path"])
        training_config = TrainingConfig.from_mapping(values)
        plan, report = run_with_plan_gate(run_training, training_config, run_enabled=RUN_TRAINING)
        summary = {
            "status": report["status"] if report else "plan_validated_training_disabled",
            "package_members_verified": audit["validated_member_count"],
            "manifest_matches": audit["manifest_matches"],
            "folds_match": audit["folds_match"],
            "remote_archive_bytes_verified": audit["remote_archive_bytes_verified"],
            "mode": plan["mode"], "fold": plan["fold"], "seed": RUN_CONFIG["seed"],
            "training_studies": plan["training_study_count"],
            "validation_studies": plan["validation_study_count"],
            "planned_steps": RUN_CONFIG["expected_steps"],
            "training_enabled": RUN_TRAINING,
        }
        if report:
            summary.update({
                "steps_completed": report["steps_completed"],
                "per_target_auc": report["per_target_auc"],
                "macro_auc": report["macro_auc"],
                "missing_class_targets": report["missing_class_targets"],
                "runtime": report["runtime"],
                "interpretation_limit": report["interpretation_limit"],
            })
        print(json.dumps(summary, sort_keys=True, allow_nan=False))
    except Exception as error:
        reason = getattr(error, "reason_code", None)
        if reason not in SAFE_REASON_CODES:
            candidate = str(error)
            reason = candidate if candidate in SAFE_REASON_CODES else "runtime_failed"
        event = {"status": "training_blocked", "reason_code": reason}
        diagnostics = safe_failure_diagnostics(error)
        if diagnostics is not None:
            event["diagnostics"] = diagnostics
        print(json.dumps(event, sort_keys=True))
        raise RuntimeError(reason) from None

if __name__ == "__main__":
    main()
